In [1]:
# Imports

import setuptools
import distutils

import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
import time
import collections
import warnings

warnings.filterwarnings('ignore')

print(f"TensorFlow  : {tf.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print(f"MediaPipe   : {mp.__version__}")


TensorFlow  : 2.16.2
OpenCV      : 4.11.0
MediaPipe   : 0.10.21


In [2]:
# Intial config 

# ── Paths ──────────────────────────────────────────────────────────────────
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
MODELS_DIR = PROJECT_DIR / "models"
BEST_MODEL_NAME_FILE = MODELS_DIR / "best_model_name.txt"

# ── Feature pipeline (must mirror data_preprocessing.ipynb exactly) ────────
NUM_KEYPOINTS  = 33
COORDS_PER_KP  = 4          # x, y, z, visibility
WINDOW_SIZE    = 30          # frames per inference window
FEATURE_DIM    = 166         # 132 kp + 33 vel + 1 aspect ratio
VISIBILITY_THR = 0.5         # same as preprocessing

# ── Inference controls ─────────────────────────────────────────────────────
FALL_THRESHOLD       = 0.5   # sigmoid probability above which a window = FALL
CONFIRM_CONSECUTIVE  = 3     # number of consecutive FALL windows before alert
DISPLAY_WIDTH        = 960
DISPLAY_HEIGHT       = 540

# ── Colours (BGR) ──────────────────────────────────────────────────────────
COLOR_NORMAL = (0, 200, 0)    # green  – no fall
COLOR_ALERT  = (0, 0, 220)    # red    – fall confirmed
COLOR_WARN   = (0, 165, 255)  # orange – fall detected, not yet confirmed

In [3]:
# Test - can delete later

import os

print("MODELS_DIR:", MODELS_DIR)
print("\nFiles in models folder:")

if os.path.exists(MODELS_DIR):
    for f in os.listdir(MODELS_DIR):
        print(f)
else:
    print("Models folder does not exist!")

MODELS_DIR: /Users/wongjoee/Desktop/GitHub/vision-based-fall-detection/models

Files in models folder:
bilstm_final.keras
best_model_name.txt


In [4]:
#Test 2 - can delete later 

print("BEST_MODEL_NAME_FILE:", BEST_MODEL_NAME_FILE)
print("File exists:", os.path.exists(BEST_MODEL_NAME_FILE))

if os.path.exists(BEST_MODEL_NAME_FILE):
    with open(BEST_MODEL_NAME_FILE, "r") as f:
        print("Selected model name:", repr(f.read().strip()))

BEST_MODEL_NAME_FILE: /Users/wongjoee/Desktop/GitHub/vision-based-fall-detection/models/best_model_name.txt
File exists: True
Selected model name: 'BiLSTM'


In [5]:
import mediapipe as mp

print("MediaPipe version:", mp.__version__)
print("MediaPipe location:", mp.__file__)
print("Has solutions:", hasattr(mp, "solutions"))

MediaPipe version: 0.10.21
MediaPipe location: /Users/wongjoee/Desktop/GitHub/vision-based-fall-detection/.venv/lib/python3.12/site-packages/mediapipe/__init__.py
Has solutions: True


In [6]:
# Load best model + initialise MediaPipe

# Load best model (saved by model_evaluation.ipynb)
with open(BEST_MODEL_NAME_FILE, "r") as f:
    best_model_name = f.read().strip()

model_path = f"{MODELS_DIR}/{best_model_name.lower()}_final.keras"
model = tf.keras.models.load_model(model_path)
print(f"Loaded model : {best_model_name}")
print(f"Input shape  : {model.input_shape}")

# MediaPipe BlazePose — identical settings to data_preprocessing.ipynb
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_styles  = mp.solutions.drawing_styles

pose_detector = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    min_detection_confidence=0.3,
    min_tracking_confidence=0.3
)
print("BlazePose initialised.")

Loaded model : BiLSTM
Input shape  : (None, 30, 166)
BlazePose initialised.


I0000 00:00:1791799761.037557 1806530 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 91.7), renderer: Apple M4


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1791799761.150348 1809088 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791799761.169537 1809090 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [7]:
#Feature extraction helpers

def extract_keypoints(frame_rgb):
    """Run BlazePose on one RGB frame. Returns (kp_vector [132,], detected bool)."""
    results = pose_detector.process(frame_rgb)
    if results.pose_landmarks is None:
        return np.zeros(NUM_KEYPOINTS * COORDS_PER_KP, dtype=np.float32), False, None
    kp_vector = []
    for lm in results.pose_landmarks.landmark:
        kp_vector.extend([lm.x, lm.y, lm.z, lm.visibility])
    return np.array(kp_vector, dtype=np.float32), True, results.pose_landmarks


def normalize_keypoints(kp_vector):
    """Person-relative bounding box normalisation — identical to preprocessing."""
    kp = kp_vector.reshape(NUM_KEYPOINTS, COORDS_PER_KP).copy()
    visible_mask = kp[:, 3] >= VISIBILITY_THR
    if visible_mask.sum() < 2:
        return kp.flatten()
    xs = kp[visible_mask, 0]
    ys = kp[visible_mask, 1]
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    scale = max(x_max - x_min, y_max - y_min, 1e-6)
    kp[:, 0] = (kp[:, 0] - x_min) / scale
    kp[:, 1] = (kp[:, 1] - y_min) / scale
    return kp.flatten()


def compute_aspect_ratio(kp_vector):
    """Body bounding box aspect ratio (height / width) — identical to preprocessing."""
    kp = kp_vector.reshape(NUM_KEYPOINTS, COORDS_PER_KP)
    visible_mask = kp[:, 3] >= VISIBILITY_THR
    if visible_mask.sum() < 2:
        return 1.0
    xs = kp[visible_mask, 0]
    ys = kp[visible_mask, 1]
    width  = xs.max() - xs.min()
    height = ys.max() - ys.min()
    if width < 1e-6:
        return 1.0
    return float(min(height / width, 5.0))


def build_feature_vector(kp_norm, prev_kp_norm):
    """Assemble the 166-dim vector for one frame given current and previous keypoints."""
    y_indices = [i * COORDS_PER_KP + 1 for i in range(NUM_KEYPOINTS)]
    if prev_kp_norm is None:
        vel = np.zeros(NUM_KEYPOINTS, dtype=np.float32)
    else:
        vel = (kp_norm[y_indices] - prev_kp_norm[y_indices]).astype(np.float32)
    ar = compute_aspect_ratio(kp_norm)
    return np.concatenate([kp_norm, vel, [ar]]).astype(np.float32)

In [8]:
# Overlay

def draw_overlay(frame, status, prob, fall_count, fps):
    """
    Draw status banner, probability bar, fall counter and FPS onto the frame.
    status : "NORMAL" | "WARNING" | "FALL DETECTED"
    """
    h, w = frame.shape[:2]

    # ── Status banner ──────────────────────────────────────────────────────
    if status == "FALL DETECTED":
        color = COLOR_ALERT
    elif status == "WARNING":
        color = COLOR_WARN
    else:
        color = COLOR_NORMAL

    cv2.rectangle(frame, (0, 0), (w, 55), color, -1)
    cv2.putText(frame, status, (12, 38),
                cv2.FONT_HERSHEY_DUPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)

    # ── Probability bar ────────────────────────────────────────────────────
    bar_x, bar_y, bar_w, bar_h = 12, 68, 260, 18
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (60, 60, 60), -1)
    fill = int(bar_w * prob)
    bar_color = COLOR_ALERT if prob >= FALL_THRESHOLD else COLOR_NORMAL
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + fill, bar_y + bar_h), bar_color, -1)
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (200, 200, 200), 1)
    cv2.putText(frame, f"Fall prob: {prob:.2f}", (bar_x, bar_y + bar_h + 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (230, 230, 230), 1, cv2.LINE_AA)

    # ── Fall event counter ─────────────────────────────────────────────────
    cv2.putText(frame, f"Fall events: {fall_count}", (12, bar_y + bar_h + 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (230, 230, 230), 1, cv2.LINE_AA)

    # ── FPS ────────────────────────────────────────────────────────────────
    cv2.putText(frame, f"FPS: {fps:.1f}", (w - 110, h - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1, cv2.LINE_AA)

    return frame

In [9]:
# Test camera input - can delete later 

import cv2

for i in range(5):
    cap_test = cv2.VideoCapture(i, cv2.CAP_AVFOUNDATION)

    if cap_test.isOpened():
        ret, frame = cap_test.read()
        if ret:
            print(f"Camera index {i}: working, frame size = {frame.shape}")
        else:
            print(f"Camera index {i}: opened, but could not read a frame")
    else:
        print(f"Camera index {i}: unavailable")

    cap_test.release()

Camera index 0: working, frame size = (1080, 1920, 3)
Camera index 1: unavailable
Camera index 2: unavailable
Camera index 3: unavailable
Camera index 4: unavailable


OpenCV: out device of bound (0-0): 1
OpenCV: camera failed to properly initialize!
[ WARN:0@39.628] global cap.cpp:480 open VIDEOIO(AVFOUNDATION): backend is generally available but can't be used to capture by index
OpenCV: out device of bound (0-0): 2
OpenCV: camera failed to properly initialize!
[ WARN:0@39.650] global cap.cpp:480 open VIDEOIO(AVFOUNDATION): backend is generally available but can't be used to capture by index
OpenCV: out device of bound (0-0): 3
OpenCV: camera failed to properly initialize!
[ WARN:0@39.692] global cap.cpp:480 open VIDEOIO(AVFOUNDATION): backend is generally available but can't be used to capture by index
OpenCV: out device of bound (0-0): 4
OpenCV: camera failed to properly initialize!
[ WARN:0@39.714] global cap.cpp:480 open VIDEOIO(AVFOUNDATION): backend is generally available but can't be used to capture by index


In [10]:
# Real time detection loop

# Try the MacBook camera first
VIDEO_SOURCE = 0
cap = cv2.VideoCapture(VIDEO_SOURCE, cv2.CAP_AVFOUNDATION)

ret, test_frame = cap.read()

if not ret:
    cap.release()

    # Fall back to the other available camera
    VIDEO_SOURCE = 1
    cap = cv2.VideoCapture(VIDEO_SOURCE, cv2.CAP_AVFOUNDATION)

    ret, test_frame = cap.read()

if not ret:
    cap.release()
    raise RuntimeError("No camera could provide a video frame.")

print(f"Using camera index: {VIDEO_SOURCE}")

# ── Rolling buffer: keeps last WINDOW_SIZE feature vectors ────────────────
frame_buffer  = collections.deque(maxlen=WINDOW_SIZE)
prev_kp_norm  = None    # for velocity computation

# ── State tracking ────────────────────────────────────────────────────────
consecutive_falls = 0   # consecutive windows predicted as fall
fall_event_count  = 0   # total confirmed fall events
current_prob      = 0.0
status            = "NORMAL"
in_fall_event     = False

# ── FPS counter ───────────────────────────────────────────────────────────
fps_timer   = time.time()
fps_counter = 0
fps_display = 0.0

print("Starting real-time detection. Press 'q' to quit.")
print(f"Model: {best_model_name}  |  Window: {WINDOW_SIZE} frames  |  Confirm threshold: {CONFIRM_CONSECUTIVE} windows")

while True:
    ret, frame = cap.read()
    if not ret:
        print("End of video / camera disconnected.")
        break

    frame = cv2.resize(frame, (DISPLAY_WIDTH, DISPLAY_HEIGHT))
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # ── Pose estimation ───────────────────────────────────────────────────
    kp_raw, detected, landmarks = extract_keypoints(frame_rgb)

    if detected:
        kp_norm = normalize_keypoints(kp_raw)
        fv = build_feature_vector(kp_norm, prev_kp_norm)
        prev_kp_norm = kp_norm
    else:
        fv = np.zeros(FEATURE_DIM, dtype=np.float32)
        prev_kp_norm = None

    frame_buffer.append(fv)

    # ── Draw skeleton when pose is detected ───────────────────────────────
    if detected and landmarks is not None:
        mp_drawing.draw_landmarks(
            frame,
            landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing.DrawingSpec(
                color=(255, 220, 100), thickness=2, circle_radius=3),
            connection_drawing_spec=mp_drawing.DrawingSpec(
                color=(180, 180, 180), thickness=1)
        )

    # ── Run inference only when buffer is full (30 frames) ────────────────
    if len(frame_buffer) == WINDOW_SIZE:
        window = np.array(frame_buffer, dtype=np.float32)   # (30, 166)
        window_input = window[np.newaxis, ...]               # (1, 30, 166)

        t0 = time.time()
        pred_prob = float(model.predict(window_input, verbose=0)[0][0])
        latency_ms = (time.time() - t0) * 1000

        current_prob = pred_prob
        is_fall = pred_prob >= FALL_THRESHOLD

        if is_fall:
            consecutive_falls += 1
            if consecutive_falls >= CONFIRM_CONSECUTIVE:
                status = "FALL DETECTED"
                if not in_fall_event:
                    fall_event_count += 1
                    in_fall_event = True
            else:
                status = "WARNING"
        else:
            consecutive_falls = 0
            status = "NORMAL"
            in_fall_event = False

    # ── FPS ───────────────────────────────────────────────────────────────
    fps_counter += 1
    elapsed = time.time() - fps_timer
    if elapsed >= 1.0:
        fps_display = fps_counter / elapsed
        fps_counter = 0
        fps_timer   = time.time()

    # ── Draw overlay ──────────────────────────────────────────────────────
    frame = draw_overlay(frame, status, current_prob, fall_event_count, fps_display)

    cv2.imshow("Fall Detection — Real Time", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        print("Quit by user.")
        break
    elif key == ord('r'):
        # Press 'r' to reset all counters mid-session
        frame_buffer.clear()
        consecutive_falls = 0
        fall_event_count  = 0
        in_fall_event     = False
        current_prob      = 0.0
        status            = "NORMAL"
        print("Counters reset.")

cap.release()
cv2.destroyAllWindows()

# Close the pose detector only if it is still open
if getattr(pose_detector, "_graph", None) is not None:
    pose_detector.close()

print(f"\nSession ended. Total fall events detected: {fall_event_count}")

Using camera index: 0
Starting real-time detection. Press 'q' to quit.
Model: BiLSTM  |  Window: 30 frames  |  Confirm threshold: 3 windows


W0000 00:00:1791799773.758906 1809093 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Quit by user.

Session ended. Total fall events detected: 16


In [17]:
# Run to disable webcam

if 'cap' in globals() and cap is not None:
    cap.release()

cv2.destroyAllWindows()

print("Webcam released and preview window closed.")


Webcam released and preview window closed.
